# Passive source example

<a target="_blank" href="https://colab.research.google.com/github/DASDAE/ctemps_tutorial/blob/master/05_passive_ex.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

#### Useful links: 
* [Colab link](https://colab.research.google.com/github/DASDAE/ctemps_tutorial/blob/master/05_passive_ex.ipynb)
* [DASCore documentation](https://dascore.org)



In [ ]:
## %%capture

# First ensure DASCore is installed. If not, install and restart the kernel.
try:
    import dascore as dc
except ImportError:
    !pip install dascore
    !pip install ipympl
    # restart kernel
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True) #automatically restarts kernel

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# helper function to flip patch along the distance axis
def flip_and_patchify(patch):
    """
    flips the input patch along the distance axis, extracts the data, then creates a new patch from the flipped input
    Using this because the flip function within dascore is not working as expected currently
    
    Parameters
    ----------
    patch: DASCore Patch
        
    data_name: String
        The name of the input 2d array
    fs: Int
        Original sampling frequency of raw data

    Returns
    -------
    pa: DASCore Patch
        The Patch format of the input array for use with DASCore
    """ 
    #attributes/metadata for patch
    attrs = patch.attrs

    dat = patch.flip("distance").data #flip data array here

    time_start = patch.coords.min("time")
    time_step = dc.to_timedelta64(patch.coords.step("time"))
    time = time_start + np.arange(dat.shape[0]) * time_step

    distance_start = patch.coords.min("distance")
    distance_step = patch.coords.step("distance")
    distance = distance_start + np.arange(dat.shape[1]) * distance_step

    coords = dict(time=time, distance = distance)

    dims = ('time', 'distance')

    pa = dc.Patch(data = dat, coords = coords, attrs = attrs, dims = dims)
    return pa

## Load in spool

In [ ]:
spool = dc.spool('/Users/nikhil_punithan/Desktop/PhD/CTEMPS26/ctemps_tutorial/Test_data') # create a spool from data directory
patch = spool[0] # isolate the first patch from 
patch.get_coord # print out patch coords

## Raw data exploration

In [ ]:
patch.viz.waterfall(cmap = 'seismic') # plot the raw data

#### Some things we want to fix:
- Data values are in velocity
- There aren't any discernable signals

In [ ]:
patch_sr = patch.velocity_to_strain_rate() # transform the data from velocity to strain rate
patch_dt1 = patch_sr.detrend("time") # detrend along time axis
patch_dt2 = patch_dt1.detrend("distance") # detrend along space axis
patch_dt2.viz.waterfall(cmap = 'seismic')

In [ ]:
## windowed patch
patch_sel = patch_dt2.select(time = ('2022-01-15T18:19:13', '2022-01-15T18:19:14'))
patch_sel.viz.waterfall(cmap = 'seismic')

### Exercise 1 (data filtering and selection)

- Zoom into different portions of data using the [select](https://dascore.org/api/dascore/proc/coords/select.html) function

- Bandpass using the [pass_filter](https://dascore.org/api/dascore/proc/filter/pass_filter.html) function

In [ ]:
### test your own code here

### Selecting sub-windows

In [ ]:
## right sided moveout
patch_sel_r = patch_sel.select(distance = (500, 970))
patch_sel_r.viz.waterfall(cmap='seismic')

In [ ]:
## left sided moveout
patch_sel_l = patch_sel.select(distance = (280, 500))
patch_sel_l.viz.waterfall(cmap = 'seismic')

#### Taking fourier transforms
- We can visualize the data in different ways using the [dft]() function

In [ ]:
## taking an f-k transform
padft = patch_sel_r.dft(patch_sel_r.dims)

padft.abs().viz.waterfall(cmap = 'cividis')

In [ ]:
## taking an f-k transform
padft2 = patch_sel_l.dft(patch_sel_l.dims)

padft2.abs().viz.waterfall(cmap = 'cividis')

Try zooming in on different quadrants using the [select](https://dascore.org/api/dascore/proc/coords/select.html) function

## Surface wave dispersion

- [Seismic wave types](https://www.geometrics.com/community/general-seismograph/what-are-the-different-types-of-seismic-waves/)

- [Multichannel analysis of surface waves (MASW)](https://www.masw.com/WhatisMASW.html)

In [ ]:
## right sided dispersion
dsp = patch_sel_r.dispersion_phase_shift(np.arange(100,3000,1),
        approx_resolution=0.1,approx_freq=[1,40])

ax = dsp.viz.waterfall(cmap = 'magma')
plt.ylim(100, 3000)

plt.gcf().axes[-1].set_ylabel("amplitude")

In [ ]:
## left sided dispersion (shouldn't work properly)
dsp_l = patch_sel_l.dispersion_phase_shift(np.arange(100,3000,1),
        approx_resolution=0.1,approx_freq=[1,40])

ax = dsp_l.viz.waterfall(cmap = 'magma')
plt.ylim(100, 3000)

plt.gcf().axes[-1].set_ylabel("amplitude")

In [ ]:
## flipping the patch
f_patch_sel_l = flip_and_patchify(patch_sel_l)
f_patch_sel_l.viz.waterfall(cmap='seismic')

In [ ]:
## left sided dispersion (corrected)
f_dsp_l = f_patch_sel_l.dispersion_phase_shift(np.arange(100, 3000,1),
        approx_resolution=.1,approx_freq=[1,40])

ax = f_dsp_l.viz.waterfall(cmap = 'magma')
plt.ylim(100, 3000)

plt.gcf().axes[-1].set_ylabel("amplitude")

## Bonus section, patch chunking and iteration

- Separating a large patch into smaller sections for more efficient processing
- Reforming a a large patch after breaking it into smaller sections

In [ ]:
## bringing back the original detrended data patch
spool.get_contents

In [ ]:
## chunk into 10 second windows
spoolc = spool.chunk(time=5)

### try chunking it into different lengths of time

In [ ]:
## check the number of patches in the new spool
spoolc.get_contents

### visualization of multiple patches

We will use the [sta lta](https://dascore.org/api/dascore/transform/stalta/stalta.html) function over multiple patches in the same spool

In [ ]:
## iterate through the spool with a for loop
for patch in spoolc[0:5]: # only looping through the first five patches

    patch_sr = patch.velocity_to_strain_rate()
    patch_dt1 = patch_sr.detrend("time")
    patch_dt2 = patch_dt1.detrend("distance")

    proc = patch_dt2.stalta(time = (.02, 1)) # calculate and visualize sta/lta
    proc.viz.waterfall()

### Processing over multiple patches and reforming a spool

In [ ]:
patch_list = []
## iterate through the spool with a for loop
for patch in spoolc[0:5]: # only looping through the first five patches

    patch_sr = patch.velocity_to_strain_rate()
    patch_dt1 = patch_sr.detrend("time")
    patch_dt2 = patch_dt1.detrend("distance")

    passed = patch_dt2.pass_filter(time = (5, 20)) # bandpass

    patch_list.append(passed)
    

In [ ]:
new_spool = dc.spool(patch_list) # create a spool from a list of patches
new_spool.get_contents